Generates an MLP regressor & tests it. 

In [1]:
# ------------- SET DATASET FEATURES HERE -------------- #

FILE_NAME: str = 'ethylene_methane_ds_10hz.parquet'    # Set name of dataset. Ensure it is placed in generated-data (shape of the ethylene-methane dataset is [time_s, methane_ppm, ethylene_ppm, feature1, ..., feature16])
PARQUET: bool = True                                   # Set only the filetype which corresponds to the dataset you are using as true.
CSV: bool = False

INPUTS: int = 16                                       # Number of features to be used for training the model
X_COLUMNS: list = list(range(3, 3+INPUTS))             # Column(s) to be used as features 
Y_COLUMNS: list | int = 1                              # Column(s) to be used as the target variable. Only use a list if there are multiple target variables.
T_COLUMN:  int = 0                                     # Column representing time (in seconds)
FREQUENCY: int = 10                                    # Sampling frequency of the dataset (in Hz)

In [2]:
# -------- SET DATASET HANDLING OPTIONS HERE ----------- #

WINDOW_SIZE: float = 1                                   # Size of the sliding window for creating sequences of data for training the model (in seconds)
SEED: int = 42                                         # Random seed for reproducibility of the train-test-validation split

Import & set filepaths

In [3]:
import numpy as np
import tensorflow as tf

from pathlib import Path

ROOT_DIR = Path.cwd().parent
DATA_DIR = ROOT_DIR.joinpath('data')
ONNX_MODEL_DIR = ROOT_DIR.joinpath('onnx_models')

from parquet_to_pd import parquetToDf
from csv_to_pd import csvToDf
from data_split import DataSplit

import matplotlib.pyplot as plt

from sklearn.utils import all_estimators
estimators = all_estimators(type_filter='regressor')
from sklearn.neural_network._multilayer_perceptron import MLPRegressor


Data ingestion & setting up split

In [4]:
if PARQUET:
    df = parquetToDf(FILE_NAME)
elif CSV:
    df = csvToDf(FILE_NAME)
else:
    raise ValueError("No filetype set to true. Set the filetype corresponding to the dataset being used to true.")

data = df.to_numpy()

winlength:int = int(WINDOW_SIZE * FREQUENCY)

X = data[:, X_COLUMNS]
y = data[:, Y_COLUMNS]
t = data[:, T_COLUMN]

splt = DataSplit(X, y, winlength, split=[0.7,0.15,0.15], time = t, seed = SEED, scaling=True)
print("Length of train, test, and validation sets:", len(splt.train), len(splt.test), len(splt.val))



Length of train, test, and validation sets: 292488 62676 62677


Set up param grid to search MLP parameters

In [9]:
grid:list[tuple[int,int]] = [(a,b) for a in range(80, 160, 20) for b in range (20, 80, 20)]
param_grid = [
  {'hidden_layer_sizes': grid, "n_iter_no_change": [10]}
]

Perform MLP grid search. This cell will hold up your kernel for a while!

In [ ]:
from sklearn_optimiser import SklearnOptimiser
mlp_clr = MLPRegressor()
optimizer = SklearnOptimiser(splt, mlp_clr, param_grid)
optimizer.optimize("grid search", n_jobs=4)

C:\Users\Fourt\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\Fourt\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\Fourt\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\Fourt\A

In [ ]:
print(optimizer.getOptimalParameters())

Convert sklearn model to onnx for porting

In [ ]:
from skl2onnx import to_onnx

onx = to_onnx(optimizer.getOptimalClassifier(), splt.train.to_numpy(flatten=True))

Save model to file

In [9]:
import onnx

onnx.save(onx, ONNX_MODEL_DIR.joinpath("model1.onnx"))